In [2]:
import pandas as pd

fi_df = pd.read_excel("../../data/processed/v1_clean_base.xlsx")

In [4]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score


cols = [
    "salary",
    "gender",
    "experience",
    "education_type",
    "region_code",
    "industry_code"
]

data = fi_df[cols].dropna().copy()

X = data[[
    "gender",
    "experience",
    "education_type",
    "region_code",
    "industry_code"
]]
y = data["salary"]

cat_features = ["gender", "education_type", "region_code", "industry_code"]
num_features = ["experience"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("num", "passthrough", num_features)
    ]
)

model = Pipeline([
    ("prep", preprocessor),
    ("reg", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
print("R2:", r2_score(y_test, pred))

perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=30,
    random_state=42,
    scoring="r2"
)

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

print(importance_df)

importance_df.to_excel("../../data/processed/importance.xlsx", index=False)

R2: 0.14881648750411358
          feature  importance_mean  importance_std
3     region_code         0.185840        0.011308
4   industry_code         0.040913        0.003348
0          gender         0.025018        0.002510
1      experience         0.015259        0.001889
2  education_type         0.012739        0.001829
